Inspect the data inside the truthtriplet files.

These are the upstream data files we will make the prong embed training data from.

To make the file that goes into this notebook run `run_lardata2hdf5.py`

Script found in `ubdl/larflow/larmatchnet/larmatch/`
```
python3 run_lardata2hdf5.py --input-larlite [larlite_file.root] --input-larcv [larcv_file.root] -o [outfile.h5] -tb -tri
```

In [1]:
import chart_studio as cs
import chart_studio.plotly as py
import plotly.graph_objects as go
import dash
from dash import Dash, html, dcc, Input, Output, callback
from dash.exceptions import PreventUpdate
from ctypes import c_int
import numpy as np
%load_ext autoreload  
%autoreload 2

In [2]:
# data useful for plots
import lardly
from lardly import DetectorOutline # load utility to draw TPC outline
detdata = DetectorOutline()
detlines = detdata.getlines(color=(10,10,10))

# PARTICLE LABEL COLORS
ssnetcolor = {-1:np.array((0,0,0)),     # ghost                                                                                                                                                   
              0:np.array((255,0,0)),   # electron                                                                                                                                       
              1:np.array((0,255,0)),   # gamma                                                                                                                             
              2:np.array((0,0,255)),   # muon                                                                                                                                              
              3:np.array((255,0,255)), # proton                                                                                                                                                 
              4:np.array((0,255,255)) # pion (+other mesons)
             }


ssnetnames = {-1:"ghost",
             0:"e",
             1:"gamma",
             2:"mu",
             3:"proton",
             4:"pion"}

kpcolors = {0:np.array((255,0,0)), # nu (red)
            1:np.array((0,255,0)), # track-start (green)
            2:np.array((0,0,255)), # track-end (blue)
            3:np.array((255,128,0)), # shower (orange)
            4:np.array((0,255,255)), # michel
            5:np.array((255,0,255))} # delta
kpnames = {0:"neutrino",
          1:"track-start",
          2:"track-end",
          3:"shower-start",
          4:"michel-start",
          5:"delta-start"}

# define default axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}
default_layout = go.Layout(
    title='Plot',
    width=1500,
    height=800,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)


In [3]:
#inputfiles = ["test.h5"]
inputfiles = ["test_traindata_fullfile.h5"]

In [4]:
from dlshowermodel.data.larmatchhit_hdf5_reader import LArMatchHitHDF5Dataset
MAX_NUM_SPACEPOINTS=50000
APPLY_MAX_FILTER=True
dataset = LArMatchHitHDF5Dataset(file_paths=inputfiles, 
                            max_num_spacepoints=MAX_NUM_SPACEPOINTS,
                            apply_max_filter=APPLY_MAX_FILTER)

Make entry table for list of files (len= 1 )
MAKE_ENTRY_TABLE: Loading from list of file paths
nkeys= 27 //ncols= 9  length= 3  for  test_traindata_fullfile.h5


/home/twongjirad/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
from torch.utils.data import DataLoader
BATCH_SIZE=1
SHUFFLE=False
NUM_WORKERS=1
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS, collate_fn=LArMatchHitHDF5Dataset.collate_fn)

In [ ]:
dataiter = iter(dataloader)

In [ ]:
# Get data
batchdata = next(dataiter)

In [ ]:
print(batchdata[0].keys())

In [ ]:
# Plot true/ghost score
batchindex = 0

xpos = batchdata[batchindex]['pos']
lmscore = batchdata[batchindex]['lmscores']
print("xpos: ",xpos.shape)
print(lmscore)

# def make_callback_customdata(wireimgs,xtriplets):
#     xucol = wireimgs[0][xtriplets[:,0],1]
#     xurow = wireimgs[0][xtriplets[:,0],0]
#     xvcol = wireimgs[1][xtriplets[:,1],1]
#     xvrow = wireimgs[1][xtriplets[:,1],0]
#     xycol = wireimgs[2][xtriplets[:,2],1]
#     xyrow = wireimgs[2][xtriplets[:,2],0]
#     customdata = np.stack( (xucol,xurow,xvcol,xvrow,xycol,xyrow), axis=-1)
#     return customdata

# spacepoint_customdata = make_callback_customdata(wireimgs,xtriplets)

# print("custom.shape: ",spacepoint_customdata.shape)
# hovertemplate='<b>U</b>: (%{customdata[0]},%{customdata[1]})<br>' + \
#             '<b>V</b>: (%{customdata[2]},%{customdata[3]})<br>' + \
#             '<b>Y</b>: (%{customdata[4]},%{customdata[5]})<br>' + \
#             'x: %{x}<br>' + \
#             'y: %{y}<br>' + \
#             'z: %{z}<br>'

# define a 3d scatter plot in plotly
plot_spacepoints = {
    "type":"scatter3d",
    "x": xpos[0,:],
    "y": xpos[1,:],
    "z": xpos[2,:],
    "mode":"markers",
    "name":"spacepoints",
    #"customdata":spacepoint_customdata,
    #"hovertemplate":hovertemplate,
    "marker":{"color":lmscore[0,:],"size":1.0,"opacity":0.5,"colorscale":"Bluered","cmin":0.3,"cmax":1.0},
    }


# collect the things we want to plot
# we add the spacepoint plot to the outline of the uboone detector
spacepoint_plot_list = detlines + [plot_spacepoints]

# define axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}

layout_spacepoint = go.Layout(default_layout)
layout_spacepoint.title = "True/Ghost Scores"


# make the plot using plotly go
spacepoint_fig = go.Figure(data=spacepoint_plot_list, layout=layout_spacepoint)
spacepoint_fig.show()

In [ ]:
# PLOT SSNET: Shower scores DATA
# We add the electron+photon scores together
import scipy

xpos = batchdata[0]['pos']

ssnet_logits = batchdata[0]['ssnet']
ssnet_probs = scipy.special.softmax( ssnet_logits, axis=0 )
shower_score = np.sum( ssnet_probs[:2,:], axis=0 ) # electron + photon scores

lmscore = batchdata[0]['lmscores'][0,:]

lmshower_score = shower_score*lmscore
lms_max = np.max(lmshower_score)
if lms_max>0.0:
    lmshower_score /= lms_max
print("lms_max: ",np.max(lmshower_score))
print("lms_min: ",np.min(lmshower_score))

ssnet_plots = []

ssnet_plot = {
    "type":"scatter3d",
    "x": xpos[0,:],
    "y": xpos[1,:],
    "z": xpos[2,:],
    "mode":"markers",
    "name":"lmshower",
    "marker":{"color":lmshower_score,"size":1.0,"opacity":0.5,"cmin":0.0,"cmax":1.0,"colorscale":"Bluered"}
}
ssnet_plots.append( ssnet_plot )

ssnet_plot_traces = detlines + ssnet_plots

layout = go.Layout(
    title='Shower score X LM score',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0.0, y=0, z=0.0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

ssnet_fig = go.Figure(data=ssnet_plot_traces, layout=layout)
ssnet_fig.show()

In [ ]:
## Let's cluster the ssnet points
from dlshowermodel.data.larmatchhit_hdf5_reader import LArMatchHitHDF5Dataset

# define a new dataset that loads all the points for dev
dev_dataset = LArMatchHitHDF5Dataset(file_paths=inputfiles, apply_max_filter=False)

In [ ]:
# get an entry from the dev dataset
dev_data = dev_dataset[0]
print(dev_data.keys())

In [ ]:
import scipy
def get_lmshower_score( ssnet_logits, lmprob ):
    ssnet_probs = scipy.special.softmax( ssnet_logits, axis=0 )
    shower_score = np.sum( ssnet_probs[:2,:], axis=0 ) # electron + photon scores
    lmshower_score = shower_score*lmprob
    #lms_max = np.max(lmshower_score)
    #if lms_max>0.0:
    #    lmshower_score /= lms_max
    return lmshower_score

In [ ]:
ssnet_logits = dev_data['ssnet']
lmprob = dev_data['lmscores']
print("dev ssnet_logits: ",ssnet_logits.shape)
print("dev lmscores: ",lmprob.shape)

lmshower = get_lmshower_score( ssnet_logits, lmprob )
print("lmshower.shape: ",lmshower.shape)

lmshower_filter = lmshower[0,:]>0.5

pos_showerhits = np.transpose( dev_data['pos'][:,lmshower_filter[:]], (1,0) )
print("shower hit array: ",pos_showerhits.shape)

# apply 

In [ ]:
# apply DBScan
from sklearn.cluster import DBSCAN

clustering = DBSCAN(eps=0.5, min_samples=5).fit(pos_showerhits)
#print(clustering.labels_)
clusterids = np.unique( clustering.labels_ )


# filter by size
threshold = 200
good_cids = []
for cid in clusterids:
    if cid==-1:
        continue
    npts = (clustering.labels_==cid).sum()
    if npts>threshold:
        good_cids.append(cid)
    else:
        clustering.labels_[ clustering.labels_==cid ] = -1

# relabel cluster IDs
for idx,cid in enumerate(good_cids):
    clustering.labels_[ clustering.labels_==cid ] = idx

clusterids = np.unique( clustering.labels_ )
print("number of above threshold clusters: ",len(good_cids))

In [ ]:
# lets view the clusters
import pandas
import plotly.express as px

colors = px.colors.qualitative.Plotly[:len(clusterids)]
print("number of discrete colors: ",len(colors))
ncolors = len(colors)
shower_cluster_plots = []
for cid in clusterids:
    if cid<0:
        continue
    #print(cid)
    xpos = pos_showerhits[ clustering.labels_==cid, : ]
    #print("cluster[",cid,"] array: ",xpos.shape)
    cluster_plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"C[%d]"%(cid),
    "marker":{"color":colors[cid%ncolors],"size":1.0,"opacity":0.5}
    }
    shower_cluster_plots.append( cluster_plot )

cluster_plot_traces = detlines + shower_cluster_plots

cluster_layout = go.Layout(default_layout)
cluster_layout.title='DBScan Clusters based on shower x lm scores'

cluster_fig = go.Figure(data=cluster_plot_traces, layout=cluster_layout)
cluster_fig.show()    


In [ ]:
## SDS development

# first we collect info about the clusters so we can sort by  size
cluster_idx_size = []
for cid in clusterids:
    if cid<0:
        continue
    clust_size = (clustering.labels_==cid).sum()
    cluster_idx_size.append( (clust_size, cid) )
cluster_idx_size.sort(reverse=True)
print("top three (size,index) pairs")
for csize, cidx in cluster_idx_size[:3]:
    print("index[",cidx,"] size=",csize)
cidx_maxnpts = cluster_idx_size[0][1]
cidx_maxnpts = cluster_idx_size[1][1]
#cidx_maxnpts = cluster_idx_size[-5][1]
cidx_maxnpts = 11

In [ ]:
import torch
import torch.nn.functional as F

class SemanticDistanceSampling(torch.nn.Module):
    def __init__(self, n_samples, temperature=0.1):
        super().__init__()
        self.n_samples = n_samples
        self.temperature = temperature

    def forward(self, points, features):
        """
        Semantic distance-based point sampling with assignment tracking
        
        Args:
            points: (B, N, 3) tensor of point coordinates
            features: (B, N, C) tensor of point features
            
        Returns:
            sampled_points: (B, n_samples, 3) tensor of sampled points
            sampled_features: (B, n_samples, C) tensor of sampled features
            sample_indices: (B, n_samples) indices of selected points
            assignments: (B, N) tensor mapping each original point to nearest sampled point
            assignment_distances: (B, N) semantic distances to assigned centroids
        """
        B, N, _ = points.shape
        device = points.device

        # Normalize features
        features_normalized = F.normalize(features, p=2, dim=-1)
        
        # Compute pairwise semantic distances
        distances = torch.cdist(features_normalized, features_normalized)
        
        # Initialize first point randomly
        first_idx = torch.randint(0, N, (B,), device=device)
        indices = first_idx.unsqueeze(-1)
        
        # Initialize distance mask
        mask = torch.ones((B, N), device=device)
        mask[torch.arange(B, device=device), first_idx] = 0

        # Iteratively sample points
        for i in range(1, self.n_samples):
            # Get distances to already sampled points
            dist_to_selected = distances[
                torch.arange(B, device=device).unsqueeze(-1),
                indices,
                :
            ]  # [B, i, N]
            
            # Minimum distance to any selected point
            min_dist = dist_to_selected.min(dim=1)[0]  # [B, N]
            
            # Apply temperature and masking
            scores = torch.exp(min_dist / self.temperature) * mask
            probs = scores / scores.sum(dim=-1, keepdim=True)
            
            # Sample next point
            next_idx = torch.multinomial(probs, 1)
            indices = torch.cat([indices, next_idx], dim=-1)
            
            # Update mask
            mask[torch.arange(B, device=device), next_idx.squeeze(-1)] = 0

        # Compute assignments for all points
        sampled_features = torch.gather(
            features,
            1,
            indices.unsqueeze(-1).expand(-1, -1, features.shape[-1])
        )
        
        # Compute distances to sampled points
        assignment_distances = torch.cdist(
            features_normalized,
            F.normalize(sampled_features, p=2, dim=-1)
        )  # [B, N, n_samples]
        
        # Get assignments (indices of nearest sampled point)
        assignments = torch.argmin(assignment_distances, dim=2)  # [B, N]
        min_distances = torch.min(assignment_distances, dim=2)[0]  # [B, N]
        
        # Gather sampled points
        sampled_points = torch.gather(
            points,
            1,
            indices.unsqueeze(-1).expand(-1, -1, points.shape[-1])
        )

        return {
            'sampled_points': sampled_points,          # [B, n_samples, 3]
            'sampled_features': sampled_features,      # [B, n_samples, C]
            'sample_indices': indices,                 # [B, n_samples]
            'assignments': assignments,                # [B, N]
            'assignment_distances': min_distances      # [B, N]
        }

In [ ]:
import torch
import torch.nn.functional as F

class DensityAwareSemanticSampling(torch.nn.Module):
    def __init__(self, n_samples, temperature=0.1, density_weight=0.3):
        super().__init__()
        self.n_samples = n_samples
        self.temperature = temperature
        self.density_weight = density_weight

    def compute_density(self, points):
        """Compute point density using KNN"""
        distances = torch.cdist(points, points)
        knn_dist,knn_indices = torch.topk(distances, k=min(16, distances.shape[-1]), dim=-1, largest=False)
        # above returns (values, indices) tensors and so [0] picks out only the values tensor
        # largest=True means that the smallest k values are returned
        # density = 1.0 / (torch.mean(knn_dist[..., 1:], dim=-1) + 1e-8)
        density = torch.exp( -torch.mean(knn_dist[..., 1:], dim=-1) )
        return density

    def forward(self, points, features):
        """
        Forward pass with cluster assignments
        
        Args:
            points: (N, 3) tensor of point coordinates
            features: (B, N, C) tensor of point features
            
        Returns:
            dict containing:
                sampled_points: (B, n_samples, 3) sampled points
                sampled_features: (B, n_samples, C) sampled features
                indices: (B, n_samples) indices of sampled points
                assignments: (B, N) cluster assignments for each point
                assignment_distances: (B, N) distance to assigned centroid
                weights: (B, N) sampling weights used
        """
        N, _ = points.shape
        device = points.device

        # Compute semantic distances
        features_normalized = F.normalize(features, p=2, dim=-1)
        semantic_distances = torch.cdist(features_normalized, features_normalized)
        max_dist = torch.max(semantic_distances)
        #print("semantic_distances: ",semantic_distances.shape)
        #print(semantic_distances)

        # Compute point density
        density = self.compute_density(points)
        max_density = torch.max(density)
        #density_normalized = density / density.max(dim=-1, keepdim=True)[0]
        density_row = density.unsqueeze(0).repeat(N,1) 
        density_col = density.unsqueeze(1).repeat(1,N)
        #print("density: ",density.shape)
        #print(density)
        #print(density_row)
        #print(density_col)
        # #dist_indicator = densitysq[:,:,:] > density[:,:]
        dist_indicator = density_row > density_col # element-wise
        #print(dist_indicator)
        ndist_above = torch.sum( dist_indicator, -1 ) # B, N
        #print("ndist_above: ",ndist_above)
        nonempty = ndist_above>0

        # get max dist for ndist_above==0 rows
        zero_semd = torch.max( semantic_distances[ ndist_above==0, : ], dim=-1 )[0]

        sdtemp = torch.clone( semantic_distances )
        
        dist_indicator[ndist_above==0,:] = True
        sdtemp[ dist_indicator==False ] = max_dist
        #print("semantic_distances after mod:")
        #print(sdtemp)
        delta_i = torch.min( sdtemp, dim=-1 )[0]
        #print("delta_i: ")
        #print(delta_i)

        delta_i[ ndist_above==0 ] = zero_semd
        #print("delta_i after max(dist) for empty density sets: ")
        #print(delta_i)

        sampling_score = density*delta_i
        #print("sampling_score: ")
        #print(sampling_score)

        kscores,kscores_indices = torch.topk(sampling_score, k=min(self.n_samples, sampling_score.shape[-1]), dim=-1, largest=True)
        #print("top-score indices: ",kscores_indices)

        # assign cluster index to these top indices
        top_dists = semantic_distances[ kscores_indices, : ]
        #print(top_dists)

        sampled_points   = points[ kscores_indices, : ]
        sampled_features = features[ kscores_indices, : ]
        assignments = torch.min( top_dists, dim=0 )[1]
        #print(assignments)
        
        return {
            'sampled_points': sampled_points,          # (n_samples, 3)
            'sampled_features': sampled_features,      # (n_samples, C)
            'assignments': assignments,                # (N)
            'weights': sampling_score                  # (N)
        }


In [ ]:
# work with largest cluster: get spacepoint positions and feature vectors

import torch
DEVICE="cpu"

cpoints = pos_showerhits[ clustering.labels_==cidx_maxnpts, : ]

# feats: first filter by lmshower score
cfeats  = dev_data['lmfeatures'][:,lmshower_filter[:]]
# next filter by cluster
cfeats = cfeats[:,clustering.labels_==cidx_maxnpts]
# transpose to be (N,C)
cfeats = np.transpose( cfeats, (1,0) )
print("cluster points: ",cpoints.shape)
print("cluster features: ",cfeats.shape)

cfeats_t = torch.from_numpy( cfeats ).to(DEVICE)
cpoints_t = torch.from_numpy( cpoints ).to(DEVICE)


In [ ]:
n_samples=5
sds_alg = SemanticDistanceSampling(n_samples=n_samples)
sds_results = sds_alg.forward( cpoints_t.unsqueeze(0), cfeats_t.unsqueeze(0))
print("sampled_points: ",sds_results["sampled_points"].shape)
print("sampled_feats: ",sds_results["sampled_features"].shape)
print("sampled_indices: ",sds_results["sample_indices"].shape)
print("assignment: ",sds_results["assignments"].shape)
print(torch.unique(sds_results["assignments"][0,:]))

In [ ]:
# Plot SDS result
nout_samples = sds_results["sampled_points"].shape[1]
sass = sds_results["assignments"][0,:]
feat_plot_v = []
for idx in range(nout_samples):
    feat_points = cpoints_t[sass[:]==idx].detach().cpu().numpy()
    feat_plot = {
        "type":"scatter3d",
        "x": feat_points[:,0],
        "y": feat_points[:,1],
        "z": feat_points[:,2],
        "mode":"markers",
        "name":"Feat[%d]"%(idx),
        "marker":{"color":colors[idx%ncolors],"size":2.0,"opacity":1.0}
    }
    feat_plot_v.append(feat_plot)

feat_plot_v = detlines + feat_plot_v

feat_layout = go.Layout(default_layout)
feat_layout.title='Sampled feature representations for cluster=%d'%(cidx_maxnpts)

feat_fig = go.Figure(data=feat_plot_v, layout=feat_layout)
feat_fig.show()     

In [ ]:
n_samples=5
dass_alg = DensityAwareSemanticSampling(n_samples=n_samples)
dass_results = dass_alg.forward( cpoints_t, cfeats_t )

In [ ]:
# Plot DynamicAwareSemanticSampling result
nout_samples = dass_results["sampled_points"].shape[0]
sass = dass_results["assignments"]
feat_plot_v = []
for idx in range(nout_samples):
    feat_points = cpoints_t[sass[:]==idx].detach().cpu().numpy()
    feat_plot = {
        "type":"scatter3d",
        "x": feat_points[:,0],
        "y": feat_points[:,1],
        "z": feat_points[:,2],
        "mode":"markers",
        "name":"Feat[%d]"%(idx),
        "marker":{"color":colors[idx%ncolors],"size":2.0,"opacity":0.5}
    }
    feat_plot_v.append(feat_plot)
sampled_points = dass_results["sampled_points"]
sampled_points_plot = {
    "type":"scatter3d",
    "x": sampled_points[:,0],
    "y": sampled_points[:,1],
    "z": sampled_points[:,2],
    "mode":"markers",
    "name":"Sampled",
    "marker":{"color":'rgba(0,0,0,1)',"size":5.0,"opacity":1.0}
    }
feat_plot_v.append(sampled_points_plot)

feat_plot_v = detlines + feat_plot_v

feat_layout = go.Layout(default_layout)
feat_layout.title='DynamicAwareSemanticSampling Results Cluster[%d]'%(cidx_maxnpts)

feat_fig = go.Figure(data=feat_plot_v, layout=feat_layout)
feat_fig.show()     